# Tenant Satisfaction Measures: Step 3, modelling overall satisfaction

In step 2 I found that several operational measures **move with** overall satisfaction (TP01). The problem is they also move with each other, and with things like region and survey method, so I can't tell which ones really matter just from correlations. A **multiple regression** looks at all of them at once, which lets me ask: *if everything else in the model stays the same, how much does each measure matter?*

What I'm doing in this notebook:
1. Rebuild the joined data from step 2
2. Decide which measures go into the model, and why
3. Fit a baseline model (context only) and a full model (context plus operations)
4. Check the model: overlap between measures, landlords with too much influence, and how well it predicts landlords it hasn't seen
5. Look at how far landlords sit from what the model predicts, and what that gap does and doesn't mean

Like step 2, I'm only modelling **housing associations** (195 landlords).

## 1. Load and join the data

In [1]:
import numpy as np
import pandas as pd
# statsmodels does the regression and gives me the full statistical output (p-values etc.).
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
# scikit-learn is only used here for cross-validation (section 5).
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

try:
    from google.colab import files
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
except ImportError:
    path = "../data/2025_TSM_Full_Data_v1.1_FINAL.xlsx"

In [2]:
# This is step 2 squashed into one cell: load both sheets, tidy the names, join on landlord code.
missing_codes = ["[X]", "[Z]", "[E]", "[U]"]
lcra = pd.read_excel(path, sheet_name="TSM25_LCRA_Perception", header=10, na_values=missing_codes)
mi = pd.read_excel(path, sheet_name="TSM25_Management_Info", header=9, na_values=missing_codes)

tp_pattern = r"^Proportion of respondents who report that they are satisfied.*\((TP\d{2})\)\d*$"
rename_map = {c: m for c in lcra.columns
              if pd.notna(m := pd.Series([c]).str.extract(tp_pattern).iloc[0, 0])}
rename_map.update({
    "Landlord name": "landlord", "Landlord code": "code", "Landlord type": "type",
    "Landlord group size (homes owned, LCRA and LCHO combined)": "size_band",
    "Landlord predominant region": "region", "Predominant collection method": "method",
})
tp_cols = sorted(v for v in rename_map.values() if v.startswith("TP"))
perception = lcra.rename(columns=rename_map)[
    ["landlord", "code", "type", "size_band", "region", "method"] + tp_cols]

def find_col(df, must_contain, starts_with=None):
    hits = [c for c in df.columns if all(t in c for t in must_contain)
            and (starts_with is None or c.startswith(starts_with))]
    assert len(hits) == 1, f"Expected 1 match for {must_contain}, found {len(hits)}"
    return hits[0]

mi_measures = {
    "asb_per_1000":             (["(NM01 1)"], "Number of anti-social"),
    "non_decent_pc":            (["(RP01)"], "Proportion"),
    "repairs_on_time_pc":       (["(RP02 1)"], "Proportion"),
    "emergency_on_time_pc":     (["(RP02 2)"], "Proportion"),
    "complaints_s1_per_1000":   (["(CH01 1)"], "LCRA"),
    "complaints_s1_on_time_pc": (["(CH02 1)"], "LCRA"),
    "complaints_s2_per_1000":   (["(CH01 2)"], "LCRA"),
    "complaints_s2_on_time_pc": (["(CH02 2)"], "LCRA"),
}
management = pd.DataFrame({"code": mi["Landlord code"]})
for short, (texts, start) in mi_measures.items():
    management[short] = pd.to_numeric(mi[find_col(mi, texts, start)], errors="coerce")

# Join, then keep housing associations only. .copy() so I can add columns without warnings.
df = perception.merge(management, on="code", how="inner", validate="one_to_one")
has = df[df["type"] == "PRP"].copy()
print("Housing associations:", len(has))

Housing associations: 195


## 2. Prepare the context variables

Some things affect satisfaction but have nothing to do with how well a landlord performs, so the model needs to allow for them:

- **Survey method.** Some methods are used by hardly any housing associations (one uses SMS, seven face to face). I learned that a category with only a handful of landlords gives unreliable estimates, so I kept Telephone, Internet and Postal and grouped the rest as "Other".
- **Size band** and **region.**

For categories like these, the model compares each group against a **reference group**. I chose the most common group each time (Telephone surveys, 5,000 to 40,000 homes, and the South), so every other group reads as "how different is this from a typical landlord".

In [3]:
# .where() keeps the method if it's one of the three big ones, and replaces it with "Other" if not.
has["method_group"] = has["method"].where(
    has["method"].isin(["Telephone", "Internet", "Postal"]), "Other")

# Making these "Categorical" lets me set the order. statsmodels uses the first one as the reference group.
has["method_group"] = pd.Categorical(has["method_group"],
                                     ["Telephone", "Internet", "Postal", "Other"])
has["size_band"] = pd.Categorical(has["size_band"], ["5,000-40,000", "<5,000", ">40,000"])
has["region"] = pd.Categorical(has["region"], ["South", "North", "Midlands", "London", "Mixed"])

has["method_group"].value_counts()

method_group
Telephone    148
Internet      22
Other         17
Postal         8
Name: count, dtype: int64

## 3. Choose the operational measures

Something I read about regression: it struggles when two predictors carry the same information, because it can't tell which one deserves the credit. So I checked how much my candidate measures overlap before putting them in.

In [4]:
# Correlations between all the candidate measures, to spot pairs that overlap.
candidates = ["complaints_s2_per_1000", "complaints_s1_per_1000", "non_decent_pc",
              "repairs_on_time_pc", "emergency_on_time_pc",
              "complaints_s1_on_time_pc", "complaints_s2_on_time_pc"]
has[candidates].corr(method="spearman").round(2)

,complaints_s2_per_1000,complaints_s1_per_1000,non_decent_pc,repairs_on_time_pc,emergency_on_time_pc,complaints_s1_on_time_pc,complaints_s2_on_time_pc
complaints_s2_per_1000,1.00,0.77,0.34,-0.19,-0.08,-0.25,-0.37
complaints_s1_per_1000,0.77,1.00,0.28,-0.19,-0.15,-0.23,-0.30
non_decent_pc,0.34,0.28,1.00,-0.08,-0.25,-0.22,-0.24
repairs_on_time_pc,-0.19,-0.19,-0.08,1.00,0.31,0.02,0.19
emergency_on_time_pc,-0.08,-0.15,-0.25,0.31,1.00,0.21,0.17
complaints_s1_on_time_pc,-0.25,-0.23,-0.22,0.02,0.21,1.00,0.65
complaints_s2_on_time_pc,-0.37,-0.30,-0.24,0.19,0.17,0.65,1.00


**What I decided:**

- **Stage 1 and stage 2 complaint volumes** are strongly linked (0.77), which makes sense: landlords with more complaints get more escalations. I kept **stage 2**, because step 2 showed it has the stronger link with satisfaction.
- **Emergency repairs on time** overlaps with non-emergency repairs, and it's close to 100% for most landlords anyway, so I only kept **non-emergency repairs on time**.
- **Stage 1 and stage 2 complaints answered on time** are basically the same habit, so I kept **stage 1**, which covers far more complaints.
- **ASB** and the **repair target** showed no link with satisfaction in step 2, so I left them out.

In [5]:
# My four operational measures, and the context written the way statsmodels likes it.
# C(...) tells statsmodels to treat a column as categories rather than numbers.
operations = ["complaints_s2_per_1000", "non_decent_pc",
              "repairs_on_time_pc", "complaints_s1_on_time_pc"]
context = "C(method_group) + C(size_band) + C(region)"

# I dropped landlords missing any of these, so both models use exactly the same landlords
# (otherwise comparing them wouldn't be fair).
model_data = has.dropna(subset=["TP01"] + operations).copy()
print("Landlords in the model:", len(model_data))

Landlords in the model: 194


## 4. Fit the models

I built two models:
- **Baseline:** satisfaction explained by context alone (method, size, region).
- **Full:** context plus my four operational measures.

Comparing them shows how much the operational measures add on top of context.

I used `cov_type="HC3"`, which gives **robust standard errors**. As I understand it, normal standard errors assume the model's errors are equally spread out for every landlord, which is rarely true with real data. Robust errors don't make that assumption, so the p-values and confidence intervals are more trustworthy. They don't change the coefficients themselves.

In [6]:
# The formula reads "TP01 explained by ...". .fit() does the actual regression.
baseline = smf.ols(f"TP01 ~ {context}", data=model_data).fit(cov_type="HC3")
full = smf.ols(f"TP01 ~ {' + '.join(operations)} + {context}", data=model_data).fit(cov_type="HC3")

print(f"Baseline R-squared: {baseline.rsquared:.3f}")
print(f"Full R-squared:     {full.rsquared:.3f}  (adjusted {full.rsquared_adj:.3f})")

Baseline R-squared: 0.317
Full R-squared:     0.589  (adjusted 0.559)


**R-squared** is the share of the differences in satisfaction between landlords that the model accounts for. Context alone gets about a third (0.32), and adding the four operational measures takes it to almost 60% (0.59). That's a lot more than I expected.

*Adjusted* R-squared gives a small penalty for each extra variable. Adding any variable always nudges plain R-squared up a bit, even a useless one, so the adjusted version is a fairer check.

In [7]:
# The full statsmodels summary is a lot to read, so I wrote a function for a simpler table.
def tidy(result):
    """A readable coefficient table: estimate, 95% confidence interval and p-value."""
    ci = result.conf_int()
    table = pd.DataFrame({"coefficient": result.params, "95% CI low": ci[0],
                          "95% CI high": ci[1], "p-value": result.pvalues})
    # Turn statsmodels' names like "C(region)[T.London]" into "region: London".
    table.index = (table.index.str.replace(r"C\((\w+)\)\[T\.(.+)\]", r"\1: \2", regex=True))
    return table.round(3)

tidy(full)

,coefficient,95% CI low,95% CI high,p-value
Intercept,61.264,50.850,71.679,0.000
complaints_s2_per_1000,-0.490,-0.605,-0.374,0.000
non_decent_pc,-1.489,-2.582,-0.396,0.008
repairs_on_time_pc,0.169,0.077,0.261,0.000
complaints_s1_on_time_pc,0.088,0.014,0.162,0.020
method_group: Internet,-1.503,-5.532,2.525,0.465
method_group: Postal,1.998,-3.355,7.351,0.464
method_group: Other,1.611,-2.699,5.920,0.464
"size_band: <5,000",-0.085,-2.398,2.228,0.943
"size_band: >40,000",-2.757,-5.560,0.047,0.054


**How I'm reading the coefficients.** Each one is how much overall satisfaction (in percentage points) is expected to change for a one-unit change in that measure, **with everything else in the model held the same**:

- **complaints_s2_per_1000:** each extra stage 2 complaint per 1,000 homes goes with about **0.5 points lower** satisfaction.
- **repairs_on_time_pc:** each extra percentage point of repairs done on time goes with about **0.17 points higher** satisfaction.
- **non_decent_pc:** each extra percentage point of non-decent homes goes with about **1.5 points lower** satisfaction. That sounds big, but most housing associations are between 0% and 0.4%, so in practice it doesn't make much difference.
- **complaints_s1_on_time_pc:** each extra percentage point of stage 1 complaints answered on time goes with about **0.09 points higher** satisfaction.
- **region: London** is around **7.5 points lower** than the South, even after allowing for performance.

A **p-value** under 0.05 is the usual line for "probably not just chance". The **95% confidence interval** is the range the true value is likely to be in. If it doesn't cross zero, I can be fairly confident about which direction the effect goes.

### Comparing measures fairly

I wanted to know which measure matters *most*, but the coefficients are in different units (complaints per 1,000 homes against percentages), so I can't just compare their sizes. A fairer way: what difference does a **typical gap** make? I used the gap between a landlord at the 25th percentile and one at the 75th (the interquartile range).

In [8]:
# The 25th and 75th percentile of each measure, and what the model says moving between them is worth.
typical = pd.DataFrame({
    "25th percentile": model_data[operations].quantile(0.25),
    "75th percentile": model_data[operations].quantile(0.75),
})
typical["effect of moving 25th to 75th (points)"] = (
    (typical["75th percentile"] - typical["25th percentile"]) * full.params[operations])
typical.round(1)

,25th percentile,75th percentile,effect of moving 25th to 75th (points)
complaints_s2_per_1000,6.0,14.0,-3.9
non_decent_pc,0.0,0.4,-0.6
repairs_on_time_pc,73.6,87.1,2.3
complaints_s1_on_time_pc,80.9,98.2,1.5


So stage 2 complaints matter most: going from a landlord with few escalations to one with lots goes with about 4 points lower satisfaction (3.9). Repairs on time is second (2.3 points).

## 5. Check the model

### Overlap between predictors (VIF)

The **variance inflation factor** measures how much each predictor overlaps with the others. 1 means no overlap at all, and the rule of thumb I found is that anything above 5 is a worry.

In [9]:
# VIF needs a constant column added (the "const"), then it works out one value per measure.
X = model_data[operations].assign(const=1.0)
pd.Series({col: variance_inflation_factor(X.values, i) for i, col in enumerate(operations)},
          name="VIF").round(2)

complaints_s2_per_1000      1.06
non_decent_pc               1.02
repairs_on_time_pc          1.03
complaints_s1_on_time_pc    1.04
Name: VIF, dtype: float64

All of them are close to 1, so the choices I made in section 3 worked: each measure carries its own information.

### Influential landlords

A few unusual landlords can drag a regression line towards them. **Cook's distance** measures how much influence each landlord has on the results. I listed the most influential ones, then refitted the model without the top three to see whether my conclusions change.

In [10]:
# get_influence() works out each landlord's influence. I also saved each landlord's residual
# (actual minus predicted), because I use it again in section 6.
influence = full.get_influence()
model_data["cooks_d"] = influence.cooks_distance[0]
model_data["residual"] = full.resid

# The five most influential landlords. I've left their names out: the point is to check
# whether a few unusual landlords are steering the results, not to single anyone out.
model_data.sort_values("cooks_d", ascending=False)[
    ["TP01"] + operations + ["residual", "cooks_d"]].head(5).round(2)

,TP01,complaints_s2_per_1000,non_decent_pc,repairs_on_time_pc,complaints_s1_on_time_pc,residual,cooks_d
321,84.1,20.6,0.0,96.9,96.6,17.08,0.09
98,44.9,11.9,5.8,82.4,57.7,-13.29,0.07
350,79.0,13.5,3.8,87.0,76.1,14.61,0.05
220,59.9,8.5,0.1,97.7,93.8,-19.98,0.05
326,97.5,0.6,0.0,100.0,100.0,14.17,0.05


In [11]:
# Refit the same model without the three most influential landlords and compare the coefficients.
top3 = model_data.sort_values("cooks_d", ascending=False).index[:3]
refit = smf.ols(f"TP01 ~ {' + '.join(operations)} + {context}",
                data=model_data.drop(top3)).fit(cov_type="HC3")

pd.DataFrame({"all landlords": full.params[operations],
              "without top 3": refit.params[operations],
              "p-value without top 3": refit.pvalues[operations]}).round(3)

,all landlords,without top 3,p-value without top 3
complaints_s2_per_1000,-0.490,-0.491,0.000
non_decent_pc,-1.489,-1.224,0.017
repairs_on_time_pc,0.169,0.156,0.000
complaints_s1_on_time_pc,0.088,0.066,0.053


The stage 2 complaints and repairs coefficients hardly move, so I'm confident those findings are **robust**. The complaints-on-time effect shrinks and becomes borderline (p = 0.053), so I'll treat that one as weaker evidence.

### Out-of-sample performance (cross-validation)

R-squared tells me how well the model fits the landlords it was built from, which flatters it. **Cross-validation** is a fairer test: split the landlords into 5 groups, build the model on 4 of them, test it on the 5th, and repeat until every group has been tested once.

In [12]:
# I reused the exact table of inputs statsmodels built (including the category columns),
# so scikit-learn is testing exactly the same model.
X_design = pd.DataFrame(full.model.exog, columns=full.model.exog_names).drop(columns="Intercept")
y = full.model.endog
X_context = X_design.drop(columns=operations)

folds = KFold(n_splits=5, shuffle=True, random_state=42)  # random_state makes results repeatable
cv_full = cross_val_score(LinearRegression(), X_design, y, cv=folds, scoring="r2")
cv_context = cross_val_score(LinearRegression(), X_context, y, cv=folds, scoring="r2")

print("Cross-validated R-squared by fold:", cv_full.round(2))
print(f"Average, full model:    {cv_full.mean():.2f}")
print(f"Average, context only:  {cv_context.mean():.2f}")

Cross-validated R-squared by fold: [0.47 0.65 0.09 0.63 0.28]
Average, full model:    0.42
Average, context only:  0.17


The model predicts unseen landlords less well than it fits the ones it was built on (0.42 against 0.59), which I gather is normal. The operational measures still add a lot over context alone (0.17). The scores jump around a lot between folds (from 0.09 to 0.65), which tells me that with under 200 landlords, a few unusual ones can sway the results.

## 6. How far do landlords sit from their predictions?

The model gives every landlord a **predicted** satisfaction score: what I'd expect from its operational performance and context. The gap between actual and predicted is the **residual**. A landlord well above its prediction is doing something the operational measures don't capture, and the same goes for one well below.

In [13]:
# Each landlord's predicted score, saved alongside the actual one (I use both for chart 4 in step 4).
model_data["predicted"] = full.predict(model_data)

# How close is the model for a typical landlord?
within_5 = (model_data["residual"].abs() <= 5).mean() * 100
beyond_10 = (model_data["residual"].abs() > 10).mean() * 100
print(f"Landlords within 5 points of their prediction: {within_5:.0f}%")
print(f"Landlords more than 10 points away:            {beyond_10:.0f}%")
print(f"Typical (median) size of the gap:              {model_data['residual'].abs().median():.1f} points")

# The spread of residuals. They should average out at about zero.
model_data["residual"].describe().round(1)

Landlords within 5 points of their prediction: 57%
Landlords more than 10 points away:            9%
Typical (median) size of the gap:              4.1 points


count    194.0
mean       0.0
std        6.1
min      -20.0
25%       -3.7
50%       -0.3
75%        4.3
max       17.1
Name: residual, dtype: float64

**57% of housing associations are within 5 points of their prediction**, and the typical gap is about 4 points. Only 9% are more than 10 points away.

Those bigger gaps are the interesting part. A landlord well below its prediction has something going on that its repairs and complaints figures don't show, such as communication or whether tenants feel listened to (notebook 9 comes back to this). I need to be careful though: the residual isn't a measure of anything specific. It's just what's left over after the model's measures, so it's a prompt for questions rather than an answer.

In [14]:
# Is the model less accurate for some sizes of landlord? Smaller landlords survey fewer tenants,
# so I'd expect their scores (and so their gaps) to be noisier.
# (Averages by group would all be zero here, because size is already in the model, so I look at
# the typical SIZE of the gap instead, ignoring whether it's above or below.)
model_data["abs_residual"] = model_data["residual"].abs()
model_data.groupby("size_band", observed=True)["abs_residual"].agg(["count", "median", "mean"]).round(1)

,count,median,mean
size_band,,,
"5,000-40,000",109,4.2,4.7
"<5,000",68,5.0,5.6
">40,000",17,2.3,2.6


As I expected, the gaps are biggest for the smallest landlords (a median of 5.0 points) and smallest for the largest (2.3), whose bigger survey samples give steadier scores. So a big gap for a small landlord needs more caution than the same gap for a large one.

## What I found in step 3

- **Context alone explains about a third** of the differences in overall satisfaction between housing associations (R² 0.32). Adding the four operational measures takes that to **almost 60%** (0.59).
- **Stage 2 complaints matter most** (about 3.9 points between a typical low and high landlord), then **repairs on time** (2.3 points). Both survive the influence check. Complaints answered on time is weaker evidence.
- On landlords it hasn't seen, the model scores **0.42**, against 0.17 for context alone.
- **57% of landlords are within 5 points of their prediction.** The ones further away, above or below, are where the operational measures miss part of the story.

**Limitations I need to be upfront about:**
- One year of data, so I'm seeing differences between landlords, not changes over time
- Correlation isn't causation: unhappy tenants may complain more, as well as complaints making tenants unhappy
- The management information is self-reported, and "on time" is measured against each landlord's own target
- Housing associations only; councils may be different

**Next:** step 4, charts that tell the story.